# Results Visualization

In [62]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from collections import defaultdict

# Plotting
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Setup
%load_ext autoreload
%autoreload 2

# Data paths
RUN_ID = "20260315_gpt_4_1_beginner_test1"
RUN_PATH = Path("outputs") / RUN_ID
DATA_PATH = Path("data")

print(f"Loading data from: {RUN_PATH}\n")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loading data from: outputs\20260315_gpt_4_1_beginner_test1



## Difficulty vs turns

In [ ]:
from utils.data import load_leetcodedataset_data, load_or_create_shuffled_data
import matplotlib.pyplot as plt

# df_train, df_test = load_leetcodedataset_data(DATA_PATH)
df_train, df_test = load_or_create_shuffled_data(DATA_PATH)
df_sample = df_train.head(100).copy()
df_sample.head()

,task_id,question_id,difficulty,tags,problem_description,starter_code,estimated_date,prompt,completion,entry_point,test,input_output,query,response
0,two-sum,1,Easy,"[Array, Hash Table]",Given an array of integers nums and an integer...,"class Solution:\n def twoSum(self, nums: Li...",2015-08-07,import random\nimport functools\nimport collec...,"class Solution:\n def twoSum(self, nums: Li...",Solution().twoSum,def check(candidate):\n assert candidate(nu...,"[{'input': 'nums = [3,3], target = 6', 'output...",You are an expert Python programmer. You will ...,"To solve this problem efficiently, we can use ..."
1,add-two-numbers,2,Medium,"[Recursion, Linked List, Math]",You are given two non-empty linked lists repre...,# Definition for singly-linked list.\n# class ...,2015-08-07,import heapq\nimport itertools\nfrom sortedcon...,# Definition for singly-linked list.\n# class ...,Solution().addTwoNumbers,def check(candidate):\n assert is_same_list...,"[{'input': 'l1 = [9,8,7], l2 = [1,2,3]', 'outp...",You are an expert Python programmer. You will ...,```python\n# Definition for singly-linked list...
2,longest-substring-without-repeating-characters,3,Medium,"[Hash Table, String, Sliding Window]","Given a string s, find the length of the longe...",class Solution:\n def lengthOfLongestSubstr...,2015-08-07,import collections\nimport string\nimport math...,class Solution:\n def lengthOfLongestSubstr...,Solution().lengthOfLongestSubstring,def check(candidate):\n assert candidate(s ...,"[{'input': 's = ""abcabcbb""', 'output': '3'}, {...",You are an expert Python programmer. You will ...,```python\nclass Solution:\n def lengthOfLo...
3,median-of-two-sorted-arrays,4,Hard,"[Array, Binary Search, Divide and Conquer]",Given two sorted arrays nums1 and nums2 of siz...,class Solution:\n def findMedianSortedArray...,2015-08-07,import random\nimport functools\nimport collec...,class Solution:\n def findMedianSortedArray...,Solution().findMedianSortedArrays,def check(candidate):\n assert candidate(nu...,"[{'input': 'nums1 = [100,200,300], nums2 = [15...",You are an expert Python programmer. You will ...,```python\nfrom typing import List\n\nclass So...
4,longest-palindromic-substring,5,Medium,"[Two Pointers, String, Dynamic Programming]","Given a string s, return the longest palindrom...",class Solution:\n def longestPalindrome(sel...,2015-08-07,import random\nimport functools\nimport collec...,class Solution:\n def longestPalindrome(sel...,Solution().longestPalindrome,def check(candidate):\n assert candidate(s ...,"[{'input': 's = ""abba""', 'output': 'abba'}, {'...",You are an expert Python programmer. You will ...,```python\nclass Solution:\n def longestPal...


In [64]:
# Configuration: group size (number of problems per group)
group_size = 10  # Change this to adjust grouping

# Prepare data for stacked plot
df_sample['group'] = (df_sample.index // group_size).astype(int)

# Count difficulty levels per group
difficulty_counts = df_sample.groupby(['group', 'difficulty']).size().unstack(fill_value=0)

# Create group labels
group_labels = [f"Problems {i*group_size}-{(i+1)*group_size-1}" for i in range(len(difficulty_counts))]

# Define colors for difficulty levels
colors = {
    'Easy': 'rgba(76, 175, 80, 0.8)',      # Green
    'Medium': 'rgba(255, 193, 7, 0.8)',    # Amber
    'Hard': 'rgba(244, 67, 54, 0.8)'       # Red
}

# Create stacked bar chart
fig = go.Figure()

for difficulty in ['Easy', 'Medium', 'Hard']:
    if difficulty in difficulty_counts.columns:
        fig.add_trace(go.Bar(
            x=group_labels,
            y=difficulty_counts[difficulty],
            name=difficulty.capitalize(),
            marker=dict(color=colors.get(difficulty, 'gray')),
            hovertemplate='<b>%{x}</b><br>' + difficulty.capitalize() + ': %{y}<extra></extra>'
        ))

fig.update_layout(
    title=f"Problem Difficulty Distribution Across Training (Grouped by {group_size} Problems)",
    xaxis_title="Training Progress",
    yaxis_title="Count",
    barmode='stack',
    height=500,
    width=1000,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=12),
    hovermode='x unified'
)

fig.show()

print(f"Difficulty Distribution (group size={group_size}):")
print(difficulty_counts)

Difficulty Distribution (group size=10):
difficulty  Easy  Hard  Medium
group                         
0              2     2       6
1              3     0       7
2              4     3       3
3              1     2       7
4              0     3       7
5              1     3       6
6              4     2       4
7              0     1       9
8              2     3       5
9              3     0       7


## Load Checkpoints Data

In [65]:
# Load memory (RL state transitions)
memory_path = RUN_PATH / "checkpoints" / "memory.csv"
df_memory = pd.read_csv(memory_path)
print(f"Loaded {len(df_memory)} transitions from memory.csv")
print(f"Columns: {df_memory.columns.tolist()}")
print(f"\nShape: {df_memory.shape}")
print(f"\nFirst few rows:")
df_memory.head()

Loaded 190 transitions from memory.csv
Columns: ['problem', 'i', 'problem_difficulty', 'last_student_level', 'last_tutor_level', 'last_action', 'last_reward', 'last_coding_score', 'action', 'reward', 'lambda', 'CODE_RUNS_REWARD', 'student_success', 'pedagogical_quality', 'student_level', 'tutor_level', 'code_runs', 'pred_reward_SOCRATIC_PROBE', 'pred_reward_CONCEPTUAL_HINT', 'pred_reward_STRUCTURAL_SCAFFOLD', 'pred_prob_SOCRATIC_PROBE', 'pred_prob_CONCEPTUAL_HINT', 'pred_prob_STRUCTURAL_SCAFFOLD']

Shape: (190, 23)

First few rows:


,problem,i,problem_difficulty,last_student_level,last_tutor_level,last_action,last_reward,last_coding_score,action,reward,...,pedagogical_quality,student_level,tutor_level,code_runs,pred_reward_SOCRATIC_PROBE,pred_reward_CONCEPTUAL_HINT,pred_reward_STRUCTURAL_SCAFFOLD,pred_prob_SOCRATIC_PROBE,pred_prob_CONCEPTUAL_HINT,pred_prob_STRUCTURAL_SCAFFOLD
0,0,0,Easy,-1,-1,NONE,-1.00,0.0,STRUCTURAL_SCAFFOLD,0.470000,...,1,3,2,True,NaN,NaN,NaN,NaN,NaN,NaN
1,0,1,Easy,3,2,STRUCTURAL_SCAFFOLD,0.47,0.1,SOCRATIC_PROBE,0.700000,...,2,3,3,True,NaN,NaN,NaN,NaN,NaN,NaN
2,0,2,Easy,3,3,SOCRATIC_PROBE,0.70,0.1,STRUCTURAL_SCAFFOLD,1.400000,...,2,2,2,True,NaN,NaN,NaN,NaN,NaN,NaN
3,1,0,Medium,-1,-1,NONE,-1.00,0.0,STRUCTURAL_SCAFFOLD,1.400000,...,2,2,2,True,NaN,NaN,NaN,NaN,NaN,NaN
4,2,0,Medium,-1,-1,NONE,-1.00,0.0,STRUCTURAL_SCAFFOLD,0.836232,...,1,3,2,True,NaN,NaN,NaN,NaN,NaN,NaN


In [66]:
# Load interaction files to understand structure
interactions_dir = RUN_PATH / "interactions"
interaction_files = sorted(interactions_dir.glob("problem_*.json"))
print(f"Found {len(interaction_files)} interaction files")

# Load one example to explore structure
with open(interaction_files[0]) as f:
    sample_interaction = json.load(f)

print(f"\nSample interaction (problem 0) has {len(sample_interaction)} entries")
print("Structure:")
for i, entry in enumerate(sample_interaction[:3]):
    print(f"  Entry {i}: {list(entry.keys())}")

Found 47 interaction files

Sample interaction (problem 0) has 5 entries
Structure:
  Entry 0: ['student_message']
  Entry 1: ['tutor_message', 'student_message', 'tutor_judgment', 'student_judgment']
  Entry 2: ['tutor_message', 'student_message', 'tutor_judgment', 'student_judgment']


# Graph 3: Tutor and Student Abstraction Level Distributions

In [67]:
def plot_abstraction_level_distribution(df, level_col, title, color_rgb, entity_name):
    """
    Create a bar chart for abstraction level distribution.
    
    Parameters:
    - df: DataFrame
    - level_col: column name (e.g., 'tutor_level', 'student_level')
    - title: chart title
    - color_rgb: color string (e.g., 'rgba(255, 182, 193, 0.8)')
    - entity_name: name of entity (e.g., 'Tutor Level', 'Student Level')
    """
    level_counts = df[level_col].value_counts().sort_index()
    
    fig = go.Figure(data=[
        go.Bar(
            x=level_counts.index.astype(str),
            y=level_counts.values,
            marker=dict(color=color_rgb, line=dict(color='darkred', width=2)),
            text=level_counts.values,
            textposition='auto',
            hovertemplate='<b>Level %{x}</b><br>Count: %{y}<extra></extra>'
        )
    ])
    
    fig.update_layout(
        title=title,
        xaxis_title=f"{entity_name} (1=Concrete, 4=Abstract)",
        yaxis_title="Frequency",
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        height=500,
        width=700,
        font=dict(size=12),
        showlegend=False
    )
    
    fig.show()
    
    print(f"\n{entity_name} Statistics:")
    print(f"Mean: {df[level_col].mean():.2f}")
    print(f"Median: {df[level_col].median():.2f}")
    print(f"Std Dev: {df[level_col].std():.2f}")
    print(f"\nCounts:\n{level_counts}")

# Plot tutor level distribution
plot_abstraction_level_distribution(
    df_memory,
    'tutor_level',
    'Judged Tutor Abstraction Levels',
    'rgba(255, 182, 193, 0.8)',
    'Tutor Level'
)


Tutor Level Statistics:
Mean: 3.08
Median: 3.00
Std Dev: 0.85

Counts:
tutor_level
1     1
2    58
3    56
4    75
Name: count, dtype: int64


In [68]:
# Plot student level distribution
plot_abstraction_level_distribution(
    df_memory,
    'student_level',
    'Judged Student Abstraction Levels',
    'rgba(135, 206, 250, 0.8)',
    'Student Level'
)


Student Level Statistics:
Mean: 2.92
Median: 3.00
Std Dev: 0.84

Counts:
student_level
1     8
2    51
3    79
4    52
Name: count, dtype: int64


In [69]:
# Create cross-tabulation (confusion matrix) of student_level vs tutor_level from automated judgments
print(f"Creating confusion matrix from {len(df_memory)} automated transitions\n")

# Create cross-tabulation
confusion_matrix = pd.crosstab(df_memory['student_level'], df_memory['tutor_level'])
print("Student Level vs Tutor Level Cross-Tabulation (Automated Judgments):")
print(confusion_matrix)
print(f"\nShape: {confusion_matrix.shape}")

# Create heatmap
fig = go.Figure(data=go.Heatmap(
    z=confusion_matrix.values,
    x=[f"Tutor Level {i}" for i in confusion_matrix.columns],
    y=[f"Student Level {i}" for i in confusion_matrix.index],
    text=confusion_matrix.values,
    texttemplate='%{text}',
    textfont={"size": 14},
    colorscale='Blues',
    colorbar=dict(title="Count"),
    hovertemplate='<b>%{y}</b><br><b>%{x}</b><br>Count: %{z}<extra></extra>'
))

fig.update_layout(
    title="Judge Alignment: Student Level vs Tutor Level (Automated Judgments)",
    xaxis_title="Tutor Abstraction Level (1=Concrete, 4=Abstract)",
    yaxis_title="Student Abstraction Level (1=Concrete, 4=Abstract)",
    height=600,
    width=750,
    font=dict(size=12),
)

fig.show()

# Print statistics
print("\n" + "="*60)
print("JUDGE ALIGNMENT ANALYSIS")
print("="*60)
print(f"\nMatching levels (where student_level == tutor_level):")
matching = sum(df_memory['student_level'] == df_memory['tutor_level'])
pct = (matching / len(df_memory)) * 100
print(f"  Count: {matching} ({pct:.1f}%)")

print(f"\nStudent ahead of tutor (student_level > tutor_level):")
ahead = sum(df_memory['student_level'] > df_memory['tutor_level'])
pct = (ahead / len(df_memory)) * 100
print(f"  Count: {ahead} ({pct:.1f}%)")

print(f"\nTutor ahead of student (tutor_level > student_level):")
tutor_ahead = sum(df_memory['tutor_level'] > df_memory['student_level'])
pct = (tutor_ahead / len(df_memory)) * 100
print(f"  Count: {tutor_ahead} ({pct:.1f}%)")

print(f"\nLevel difference (abs):")
level_diff = abs(df_memory['student_level'] - df_memory['tutor_level'])
print(f"  Mean: {level_diff.mean():.2f}")
print(f"  Median: {level_diff.median():.2f}")
print(f"  Max: {level_diff.max():.0f}")


Creating confusion matrix from 190 automated transitions

Student Level vs Tutor Level Cross-Tabulation (Automated Judgments):
tutor_level    1   2   3   4
student_level               
1              0   2   1   5
2              1  31  13   6
3              0  22  34  23
4              0   3   8  41

Shape: (4, 4)



JUDGE ALIGNMENT ANALYSIS

Matching levels (where student_level == tutor_level):
  Count: 106 (55.8%)

Student ahead of tutor (student_level > tutor_level):
  Count: 34 (17.9%)

Tutor ahead of student (tutor_level > student_level):
  Count: 50 (26.3%)

Level difference (abs):
  Mean: 0.55
  Median: 0.00
  Max: 3


# Graph 10: Pedagogical Move Distribution

In [70]:
# Get pedagogical move distribution
action_counts = df_memory['action'].value_counts()
action_order = ["SOCRATIC_PROBE", "CONCEPTUAL_HINT", "STRUCTURAL_SCAFFOLD"]
action_counts = action_counts.reindex([a for a in action_order if a in action_counts.index])

# Color mapping for actions
action_colors = {
    "SOCRATIC_PROBE": "rgba(100, 149, 237, 0.8)",        # Cornflower blue
    "CONCEPTUAL_HINT": "rgba(144, 238, 144, 0.8)",       # Light green
    "STRUCTURAL_SCAFFOLD": "rgba(255, 165, 0, 0.8)",     # Orange
}

fig = go.Figure(data=[
    go.Bar(
        x=action_counts.index,
        y=action_counts.values,
        marker=dict(
            color=[action_colors.get(action, 'gray') for action in action_counts.index],
            line=dict(color='black', width=1.5)
        ),
        text=action_counts.values,
        textposition='auto',
        hovertemplate='<b>%{x}</b><br>Count: %{y}<extra></extra>'
    )
])

fig.update_layout(
    title="Distribution of Pedagogical Actions (Training Cycle)",
    xaxis_title="Pedagogical Move",
    yaxis_title="Frequency",
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    height=500,
    width=800,
    font=dict(size=12),
    showlegend=False,
    xaxis=dict(tickangle=-15)
)

fig.show()

print(f"\nPedagogical Move Statistics:")
print(f"Total actions: {action_counts.sum()}")
print(f"\nCounts:")
for action, count in action_counts.items():
    pct = (count / action_counts.sum()) * 100
    print(f"  {action}: {count} ({pct:.1f}%)")


Pedagogical Move Statistics:
Total actions: 190

Counts:
  SOCRATIC_PROBE: 63 (33.2%)
  CONCEPTUAL_HINT: 65 (34.2%)
  STRUCTURAL_SCAFFOLD: 62 (32.6%)


# Graph 10.2: Action with Highest Predicted Reward

In [71]:
# Extract best actions from predicted rewards
pred_columns = ['pred_reward_SOCRATIC_PROBE', 'pred_reward_CONCEPTUAL_HINT', 'pred_reward_STRUCTURAL_SCAFFOLD']
df_with_preds = df_memory[pred_columns].dropna()

if len(df_with_preds) > 0:
    # Find which action had the highest predicted reward for each row
    best_actions = df_with_preds.idxmax(axis=1).str.replace('pred_reward_', '')
    best_action_counts = best_actions.value_counts()
    best_action_counts = best_action_counts.reindex([a for a in action_order if a in best_action_counts.index])
    
    # Create visualization
    fig = go.Figure(data=[
        go.Bar(
            x=best_action_counts.index,
            y=best_action_counts.values,
            marker=dict(
                color=[action_colors.get(action, 'gray') for action in best_action_counts.index],
                line=dict(color='black', width=1.5)
            ),
            text=best_action_counts.values,
            textposition='auto',
            hovertemplate='<b>%{x}</b><br>Count: %{y}<extra></extra>'
        )
    ])
    
    fig.update_layout(
        title="Distribution of Optimal Actions (Exploitation Cycle)",
        xaxis_title="Pedagogical Move",
        yaxis_title="Frequency",
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        height=500,
        width=800,
        font=dict(size=12),
        showlegend=False,
        xaxis=dict(tickangle=-15)
    )
    
    fig.show()
    
    # Print statistics
    print(f"\nHighest Predicted Reward Action Statistics:")
    print(f"Rows with predictions: {len(df_with_preds)}")
    print(f"\nCounts:")
    for action, count in best_action_counts.items():
        pct = (count / best_action_counts.sum()) * 100
        print(f"  {action}: {count} ({pct:.1f}%)")
else:
    print("No rows with predicted rewards found")


Highest Predicted Reward Action Statistics:
Rows with predictions: 164

Counts:
  SOCRATIC_PROBE: 27 (16.5%)
  CONCEPTUAL_HINT: 63 (38.4%)
  STRUCTURAL_SCAFFOLD: 74 (45.1%)


# Graph 6: Highest Predicted Action Distribution Over Training Time

In [87]:
pred_columns = ['pred_reward_SOCRATIC_PROBE', 'pred_reward_CONCEPTUAL_HINT', 'pred_reward_STRUCTURAL_SCAFFOLD']
group_size = 10  # Group every N problems

# Extract best actions from predictions
df_with_preds = df_memory[pred_columns].dropna()

if len(df_with_preds) > 0:
    best_actions = df_with_preds.idxmax(axis=1).str.replace('pred_reward_', '')
    problem_nums = df_memory.loc[best_actions.index, 'problem'].values
    
    # Group by training progress
    df_grouped = pd.DataFrame({
        'best_action': best_actions.values,
        'problem': problem_nums
    })
    df_grouped['group'] = (df_grouped['problem'] // group_size).astype(int)
    
    group_action_counts = df_grouped.groupby(['group', 'best_action']).size().unstack(fill_value=0)
    group_action_counts = group_action_counts.reindex([a for a in action_order if a in group_action_counts.columns], axis=1, fill_value=0)
    
    # Create visualization
    group_labels = [f"Problems {i*group_size}-{(i+1)*group_size-1}" for i in group_action_counts.index]
    
    fig = go.Figure()
    for action in group_action_counts.columns:
        fig.add_trace(go.Scatter(
            x=group_labels,
            y=group_action_counts[action],
            mode='lines',
            name=action,
            line=dict(width=0),
            fillcolor=action_colors.get(action, 'gray'),
            stackgroup='one',
            hovertemplate='<b>%{fullData.name}</b><br>%{x}<br>Count: %{y}<extra></extra>'
        ))
    
    fig.update_layout(
        title=f"Distribution of Optimal Actions Over Training (Grouped by {group_size} Problems)",
        xaxis_title="Training Progress",
        yaxis_title="Frequency",
        hovermode='x unified',
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        height=500,
        width=900,
        font=dict(size=11),
    )
    
    fig.show()
    
    # Print summary
    print(f"\nHighest Predicted Action Over Training Time (group size={group_size}):")
    print(f"Total groups: {len(group_action_counts)}")
    print(f"\n{group_action_counts}")
else:
    print("No rows with predicted rewards found")


Highest Predicted Action Over Training Time (group size=10):
Total groups: 5

best_action  SOCRATIC_PROBE  CONCEPTUAL_HINT  STRUCTURAL_SCAFFOLD
group                                                            
0                         0                1                   16
1                         6                9                   10
2                         2               15                   12
3                         7               20                   24
4                        12               18                   12


# Graph 1: Conversation End Reasons (Pie Chart)

In [73]:
import json
from collections import Counter

# Extract stop reasons from all interaction files
stop_reasons = []
stop_explanations = Counter()

for interaction_file in interaction_files:
    with open(interaction_file, 'r') as f:
        interactions = json.load(f)
    
    # Find the stop entry (should be last or near end)
    for entry in interactions:
        if isinstance(entry, dict) and "stop" in entry:
            stop_info = entry["stop"]
            
            # Determine the reason
            if "error" in stop_info:
                reason = "Error"
            elif "problem_finished" in stop_info:
                if stop_info["problem_finished"]:
                    reason = "Problem Solved"
                else:
                    explanation = stop_info.get("explanation", "Unknown")
                    reason = explanation
            else:
                reason = "Unknown"
            
            stop_reasons.append(reason)
            stop_explanations[reason] += 1

# Aggregate similar reasons
reason_counts = Counter(stop_reasons)
print("Stop Reasons Distribution:")
for reason, count in reason_counts.most_common():
    pct = (count / len(stop_reasons)) * 100
    print(f"  {reason}: {count} ({pct:.1f}%)")

# Create pie chart
colors = {
    "Problem Solved": "rgba(76, 175, 80, 0.8)",           # Green
    "Max number of iterations": "rgba(255, 193, 7, 0.8)", # Amber
    "Leakage detected": "rgba(244, 67, 54, 0.8)",         # Red
    "Student changed problem": "rgba(233, 30, 99, 0.8)",  # Pink
    "Error": "rgba(156, 39, 176, 0.8)",                   # Purple
}

fig = go.Figure(data=[
    go.Pie(
        labels=list(reason_counts.keys()),
        values=list(reason_counts.values()),
        marker=dict(
            colors=[colors.get(reason, 'rgba(158, 158, 158, 0.8)') for reason in reason_counts.keys()],
            line=dict(color='white', width=2)
        ),
        textposition='inside',
        textinfo='label+percent',
        hovertemplate='<b>%{label}</b><br>Count: %{value}<br>Percentage: %{percent}<extra></extra>'
    )
])

fig.update_layout(
    title="How Conversations Ended (100 Problems)",
    height=600,
    width=700,
    font=dict(size=12),
    showlegend=True
)

fig.show()

print(f"\nTotal problems analyzed: {len(stop_reasons)}")

Stop Reasons Distribution:
  Problem Solved: 37 (78.7%)
  Max number of iterations: 8 (17.0%)
  Error: 2 (4.3%)



Total problems analyzed: 47


In [74]:
# Cumulative Reward vs Turns
# Load memory fresh and compute cumulative metrics per turn
df = pd.read_csv(RUN_PATH / "checkpoints" / "memory.csv")

# Group by turn index (i) and calculate reward statistics
reward_by_turn = df.groupby('i')['reward'].agg(['mean', 'median', 'std', 'count']).reset_index()

print("Cumulative Reward by Turn Index:")
print(reward_by_turn)
print(f"\nTotal turns: {len(reward_by_turn)}")

# Create plot with mean and median lines
fig = go.Figure()

# Mean line
fig.add_trace(go.Scatter(
    x=reward_by_turn['i'],
    y=reward_by_turn['mean'],
    mode='lines+markers',
    name='Mean Reward',
    line=dict(color='rgba(255, 152, 0, 1)', width=3),
    marker=dict(size=8),
    hovertemplate='Turn %{x}: Mean Reward = %{y:.3f}<extra></extra>'
))

# Fill between mean ± std
fig.add_trace(go.Scatter(
    x=reward_by_turn['i'].tolist() + reward_by_turn['i'].tolist()[::-1],
    y=(reward_by_turn['mean'] + reward_by_turn['std']).tolist() + (reward_by_turn['mean'] - reward_by_turn['std']).tolist()[::-1],
    fill='toself',
    fillcolor='rgba(255, 152, 0, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    name='±1 Std Dev',
    hoverinfo='skip'
))

fig.update_layout(
    title="RL Reward Signal vs Conversation Turn",
    xaxis_title="Turn Index (0 = first tutor response)",
    yaxis_title="Reward (student_success + pedagogical_quality)",
    height=500,
    width=900,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=12),
    hovermode='x unified',
)

fig.show()

print(f"\nInsights:")
print(f"  Turn 0 mean reward: {reward_by_turn.iloc[0]['mean']:.3f}")
print(f"  Best turn: {reward_by_turn.loc[reward_by_turn['mean'].idxmax(), 'i']:.0f} with {reward_by_turn['mean'].max():.3f}")
print(f"  Worst turn: {reward_by_turn.loc[reward_by_turn['mean'].idxmin(), 'i']:.0f} with {reward_by_turn['mean'].min():.3f}")
print(f"  Overall trend: {'Improving' if reward_by_turn.iloc[-1]['mean'] > reward_by_turn.iloc[0]['mean'] else 'Declining'}")


Cumulative Reward by Turn Index:
   i      mean  median       std  count
0  0  0.759586    0.70  0.389916     46
1  1  0.798139    0.70  0.388227     35
2  2  0.726989    0.70  0.370172     25
3  3  0.592316    0.60  0.295479     17
4  4  0.650000    0.65  0.336650     16
5  5  0.491667    0.70  0.299874     12
6  6  0.575000    0.60  0.298861     12
7  7  0.660000    0.70  0.323866     10
8  8  0.611111    0.70  0.161589      9
9  9  0.450000    0.50  0.207020      8

Total turns: 10



Insights:
  Turn 0 mean reward: 0.760
  Best turn: 1 with 0.798
  Worst turn: 9 with 0.450
  Overall trend: Declining


# Performance Metrics vs Conversation Turns

In [75]:
# Student Success vs Turns
# Load memory fresh and group by turn index
df = pd.read_csv(RUN_PATH / "checkpoints" / "memory.csv")

# Group by turn index (i) and calculate statistics
success_by_turn = df.groupby('i')['student_success'].agg(['mean', 'median', 'std', 'count']).reset_index()

print("Student Success by Turn Index:")
print(success_by_turn)
print(f"\nTotal turns: {len(success_by_turn)}")

# Create plot with mean and median lines
fig = go.Figure()

# Mean line
fig.add_trace(go.Scatter(
    x=success_by_turn['i'],
    y=success_by_turn['mean'],
    mode='lines+markers',
    name='Mean Success',
    line=dict(color='rgba(76, 175, 80, 1)', width=3),
    marker=dict(size=8),
    hovertemplate='Turn %{x}: Mean Success = %{y:.2%}<extra></extra>'
))

# Fill between mean ± std
fig.add_trace(go.Scatter(
    x=success_by_turn['i'].tolist() + success_by_turn['i'].tolist()[::-1],
    y=(success_by_turn['mean'] + success_by_turn['std']).tolist() + (success_by_turn['mean'] - success_by_turn['std']).tolist()[::-1],
    fill='toself',
    fillcolor='rgba(76, 175, 80, 0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    name='±1 Std Dev',
    hoverinfo='skip'
))

fig.update_layout(
    title="Student Code Success vs Conversation Turn",
    xaxis_title="Turn Index (0 = first tutor response)",
    yaxis_title="Student Success Rate (% tests passed)",
    height=500,
    width=900,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=12),
    hovermode='x unified',
)

fig.update_yaxes(tickformat='.0%')
fig.show()

print(f"\nInsights:")
print(f"  Turn 0 mean success: {success_by_turn.iloc[0]['mean']:.1%}")
print(f"  Best turn: {success_by_turn.loc[success_by_turn['mean'].idxmax(), 'i']:.0f} with {success_by_turn['mean'].max():.1%}")
print(f"  Worst turn: {success_by_turn.loc[success_by_turn['mean'].idxmin(), 'i']:.0f} with {success_by_turn['mean'].min():.1%}")


Student Success by Turn Index:
   i      mean  median       std  count
0  0  0.321148     0.0  0.450539     46
1  1  0.393260     0.0  0.476179     35
2  2  0.329984     0.0  0.465685     25
3  3  0.165494     0.0  0.369784     17
4  4  0.250000     0.0  0.447214     16
5  5  0.000000     0.0  0.000000     12
6  6  0.166667     0.0  0.389249     12
7  7  0.100000     0.0  0.316228     10
8  8  0.111111     0.0  0.333333      9
9  9  0.000000     0.0  0.000000      8

Total turns: 10



Insights:
  Turn 0 mean success: 32.1%
  Best turn: 1 with 39.3%
  Worst turn: 5 with 0.0%


In [76]:
# Model Training Progress Across 100 Conversations
# Load memory fresh
df = pd.read_csv(RUN_PATH / "checkpoints" / "memory.csv")

# Configuration: group size (number of consecutive problems to group together)
group_size_problems = 5  # Number of problems to group (configurable)

# Calculate mean accuracy per problem (0-99)
accuracy_by_problem = df.groupby('problem')['student_success'].agg(['mean', 'std', 'count']).reset_index()
accuracy_by_problem.columns = ['problem', 'accuracy_mean', 'accuracy_std', 'count']

# Group problems into chunks and calculate mean accuracy per group
accuracy_by_problem['group'] = (accuracy_by_problem['problem'] // group_size_problems).astype(int)
grouped_accuracy = accuracy_by_problem.groupby('group').agg({
    'accuracy_mean': ['mean', 'std'],
    'count': 'sum'
}).reset_index()

# Flatten column names
grouped_accuracy.columns = ['group', 'mean', 'std', 'total_transitions']

# Create group labels
grouped_accuracy['label'] = grouped_accuracy['group'].apply(
    lambda g: f"Problems {g*group_size_problems}-{(g+1)*group_size_problems-1}"
)

print(f"Model Training Progress Across 100 Conversations (group size = {group_size_problems} problems):")
print(grouped_accuracy[['label', 'mean', 'std', 'total_transitions']])

# Create bar chart with error bars
fig = go.Figure(data=[
    go.Bar(
        x=grouped_accuracy['label'],
        y=grouped_accuracy['mean'],
        error_y=dict(
            type='data',
            array=grouped_accuracy['std'],
            visible=True
        ),
        marker=dict(
            color='rgba(66, 133, 244, 0.8)',
            line=dict(color='rgba(25, 103, 210, 1)', width=2)
        ),
        text=[f"{v:.1%}" for v in grouped_accuracy['mean']],
        textposition='outside',
        hovertemplate='<b>%{x}</b><br>Mean Accuracy: %{y:.2%}<br>Std Dev: %{error_y.array:.2%}<br>Transitions: %{customdata}<extra></extra>',
        customdata=grouped_accuracy['total_transitions']
    )
])

fig.update_layout(
    title=f"Model Training Progress: Student Accuracy Across 100 Conversations (Grouped by {group_size_problems} Problems)",
    xaxis_title="Training Conversation Progress",
    yaxis_title="Mean Student Success Rate",
    height=500,
    width=1000,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=12),
    showlegend=False
)

fig.update_yaxes(tickformat='.0%', range=[0, 1.0])
fig.show()

# Print summary statistics
print(f"\n{'='*60}")
print(f"Training Progress Summary (group size = {group_size_problems} problems):")
print(f"{'='*60}")
print(f"Total groups: {len(grouped_accuracy)}")
print(f"First group (early training) mean accuracy: {grouped_accuracy.iloc[0]['mean']:.1%}")
print(f"Last group (late training) mean accuracy: {grouped_accuracy.iloc[-1]['mean']:.1%}")
improvement = grouped_accuracy.iloc[-1]['mean'] - grouped_accuracy.iloc[0]['mean']
print(f"Overall improvement: {improvement:+.1%}")
best_idx = grouped_accuracy['mean'].idxmax()
print(f"Best group: {grouped_accuracy.loc[best_idx, 'label']} ({grouped_accuracy.loc[best_idx, 'mean']:.1%})")
worst_idx = grouped_accuracy['mean'].idxmin()
print(f"Worst group: {grouped_accuracy.loc[worst_idx, 'label']} ({grouped_accuracy.loc[worst_idx, 'mean']:.1%})")


Model Training Progress Across 100 Conversations (group size = 5 problems):
            label      mean       std  total_transitions
0    Problems 0-4  0.653555  0.407882                 17
1    Problems 5-9  0.427691  0.133963                 14
2  Problems 10-14  0.698593  0.289768                 10
3  Problems 15-19  0.582282  0.480181                 18
4  Problems 20-24  0.581278  0.279451                 15
5  Problems 25-29  0.749132  0.499424                 14
6  Problems 30-34  0.214933  0.200147                 32
7  Problems 35-39  0.547527  0.455555                 22
8  Problems 40-44  0.312500  0.473242                 25
9  Problems 45-49  0.142361  0.139062                 23



Training Progress Summary (group size = 5 problems):
Total groups: 10
First group (early training) mean accuracy: 65.4%
Last group (late training) mean accuracy: 14.2%
Overall improvement: -51.1%
Best group: Problems 25-29 (74.9%)
Worst group: Problems 45-49 (14.2%)


In [77]:
# Model Training Progress: Cumulative Reward Across 100 Conversations
# Load memory fresh
df = pd.read_csv(RUN_PATH / "checkpoints" / "memory.csv")

# Configuration: group size (number of consecutive problems to group together)
group_size_problems = 10  # Number of problems to group (configurable)

# Calculate mean reward per problem (0-99)
reward_by_problem = df.groupby('problem')['reward'].agg(['mean', 'std', 'count']).reset_index()
reward_by_problem.columns = ['problem', 'reward_mean', 'reward_std', 'count']

# Group problems into chunks and calculate mean reward per group
reward_by_problem['group'] = (reward_by_problem['problem'] // group_size_problems).astype(int)
grouped_reward = reward_by_problem.groupby('group').agg({
    'reward_mean': ['mean', 'std'],
    'count': 'sum'
}).reset_index()

# Flatten column names
grouped_reward.columns = ['group', 'mean', 'std', 'total_transitions']

# Create group labels
grouped_reward['label'] = grouped_reward['group'].apply(
    lambda g: f"Problems {g*group_size_problems}-{(g+1)*group_size_problems-1}"
)

print(f"Model Training Progress - Cumulative Reward (group size = {group_size_problems} problems):")
print(grouped_reward[['label', 'mean', 'std', 'total_transitions']])

# Create bar chart with error bars
fig = go.Figure(data=[
    go.Bar(
        x=grouped_reward['label'],
        y=grouped_reward['mean'],
        error_y=dict(
            type='data',
            array=grouped_reward['std'],
            visible=True
        ),
        marker=dict(
            color='rgba(255, 152, 0, 0.8)',
            line=dict(color='rgba(230, 124, 0, 1)', width=2)
        ),
        text=[f"{v:.3f}" for v in grouped_reward['mean']],
        textposition='outside',
        hovertemplate='<b>%{x}</b><br>Mean Reward: %{y:.4f}<br>Std Dev: %{error_y.array:.4f}<br>Transitions: %{customdata}<extra></extra>',
        customdata=grouped_reward['total_transitions']
    )
])

fig.update_layout(
    title=f"Model Training Progress: Cumulative Reward Across 100 Conversations (Grouped by {group_size_problems} Problems)",
    xaxis_title="Training Conversation Progress",
    yaxis_title="Mean Reward Signal",
    height=500,
    width=1000,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=12),
    showlegend=False
)

fig.show()

# Print summary statistics
print(f"\n{'='*60}")
print(f"Training Progress Summary - Reward (group size = {group_size_problems} problems):")
print(f"{'='*60}")
print(f"Total groups: {len(grouped_reward)}")
print(f"First group (early training) mean reward: {grouped_reward.iloc[0]['mean']:.4f}")
print(f"Last group (late training) mean reward: {grouped_reward.iloc[-1]['mean']:.4f}")
improvement = grouped_reward.iloc[-1]['mean'] - grouped_reward.iloc[0]['mean']
print(f"Overall change: {improvement:+.4f}")
best_idx = grouped_reward['mean'].idxmax()
print(f"Best group: {grouped_reward.loc[best_idx, 'label']} ({grouped_reward.loc[best_idx, 'mean']:.4f})")
worst_idx = grouped_reward['mean'].idxmin()
print(f"Worst group: {grouped_reward.loc[worst_idx, 'label']} ({grouped_reward.loc[worst_idx, 'mean']:.4f})")

Model Training Progress - Cumulative Reward (group size = 10 problems):
            label      mean       std  total_transitions
0    Problems 0-9  0.886103  0.281055                 31
1  Problems 10-19  0.972829  0.295595                 28
2  Problems 20-29  1.037687  0.336822                 29
3  Problems 30-39  0.747432  0.330645                 54
4  Problems 40-49  0.703785  0.315281                 48



Training Progress Summary - Reward (group size = 10 problems):
Total groups: 5
First group (early training) mean reward: 0.8861
Last group (late training) mean reward: 0.7038
Overall change: -0.1823
Best group: Problems 20-29 (1.0377)
Worst group: Problems 40-49 (0.7038)


In [78]:
# Model Training Progress: Reward Model Mean Absolute Error (MAE) Across 100 Conversations
# Load memory fresh
df = pd.read_csv(RUN_PATH / "checkpoints" / "memory.csv")

# Configuration: group size (number of consecutive problems to group together)
group_size_problems = 5  # Number of problems to group (configurable)

# Compute MAE for each transition
# MAE = |actual_reward - predicted_reward|
# For predicted reward, use the best (max) predicted reward among the three actions
pred_columns = ['pred_reward_SOCRATIC_PROBE', 'pred_reward_CONCEPTUAL_HINT', 'pred_reward_STRUCTURAL_SCAFFOLD']
df_with_preds = df[pred_columns + ['reward', 'problem']].dropna()

if len(df_with_preds) > 0:
    # Get best predicted reward for each row
    best_pred_reward = df_with_preds[pred_columns].max(axis=1)
    
    # Compute MAE
    df_with_preds['mae'] = abs(df_with_preds['reward'] - best_pred_reward)
    
    # Calculate mean MAE per problem (0-99)
    mae_by_problem = df_with_preds.groupby('problem')['mae'].agg(['mean', 'std', 'count']).reset_index()
    mae_by_problem.columns = ['problem', 'mae_mean', 'mae_std', 'count']
    
    # Group problems into chunks and calculate mean MAE per group
    mae_by_problem['group'] = (mae_by_problem['problem'] // group_size_problems).astype(int)
    grouped_mae = mae_by_problem.groupby('group').agg({
        'mae_mean': ['mean', 'std'],
        'count': 'sum'
    }).reset_index()
    
    # Flatten column names
    grouped_mae.columns = ['group', 'mean', 'std', 'total_transitions']
    
    # Create group labels
    grouped_mae['label'] = grouped_mae['group'].apply(
        lambda g: f"Problems {g*group_size_problems}-{(g+1)*group_size_problems-1}"
    )
    
    print(f"Model Training Progress - Reward Model MAE (group size = {group_size_problems} problems):")
    print(grouped_mae[['label', 'mean', 'std', 'total_transitions']])
    
    # Create bar chart with error bars
    fig = go.Figure(data=[
        go.Bar(
            x=grouped_mae['label'],
            y=grouped_mae['mean'],
            error_y=dict(
                type='data',
                array=grouped_mae['std'],
                visible=True
            ),
            marker=dict(
                color='rgba(244, 67, 54, 0.8)',
                line=dict(color='rgba(200, 30, 20, 1)', width=2)
            ),
            text=[f"{v:.4f}" for v in grouped_mae['mean']],
            textposition='outside',
            hovertemplate='<b>%{x}</b><br>Mean MAE: %{y:.5f}<br>Std Dev: %{error_y.array:.5f}<br>Transitions: %{customdata}<extra></extra>',
            customdata=grouped_mae['total_transitions']
        )
    ])
    
    fig.update_layout(
        title=f"Model Training Progress: Reward Model MAE Across 100 Conversations (Grouped by {group_size_problems} Problems)",
        xaxis_title="Training Conversation Progress",
        yaxis_title="Mean Absolute Error (Actual vs Predicted Reward)",
        height=500,
        width=1000,
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        font=dict(size=12),
        showlegend=False
    )
    
    fig.show()
    
    # Print summary statistics
    print(f"\n{'='*60}")
    print(f"Training Progress Summary - MAE (group size = {group_size_problems} problems):")
    print(f"{'='*60}")
    print(f"Total groups: {len(grouped_mae)}")
    print(f"First group (early training) mean MAE: {grouped_mae.iloc[0]['mean']:.5f}")
    print(f"Last group (late training) mean MAE: {grouped_mae.iloc[-1]['mean']:.5f}")
    improvement = grouped_mae.iloc[0]['mean'] - grouped_mae.iloc[-1]['mean']
    print(f"MAE reduction (improvement): {improvement:+.5f}")
    best_idx = grouped_mae['mean'].idxmin()
    print(f"Best group (lowest MAE): {grouped_mae.loc[best_idx, 'label']} ({grouped_mae.loc[best_idx, 'mean']:.5f})")
    worst_idx = grouped_mae['mean'].idxmax()
    print(f"Worst group (highest MAE): {grouped_mae.loc[worst_idx, 'label']} ({grouped_mae.loc[worst_idx, 'mean']:.5f})")
else:
    print("No rows with predicted rewards found")

Model Training Progress - Reward Model MAE (group size = 5 problems):
            label      mean       std  total_transitions
0    Problems 0-4  0.274365       NaN                  5
1    Problems 5-9  0.376444  0.123913                 12
2  Problems 10-14  0.304280  0.227689                  8
3  Problems 15-19  0.314350  0.155394                 17
4  Problems 20-24  0.350498  0.094098                 15
5  Problems 25-29  0.437155  0.224272                 14
6  Problems 30-34  0.451639  0.202712                 30
7  Problems 35-39  0.329428  0.131625                 21
8  Problems 40-44  0.395700  0.145872                 22
9  Problems 45-49  0.291465  0.096338                 20



Training Progress Summary - MAE (group size = 5 problems):
Total groups: 10
First group (early training) mean MAE: 0.27436
Last group (late training) mean MAE: 0.29146
MAE reduction (improvement): -0.01710
Best group (lowest MAE): Problems 0-4 (0.27436)
Worst group (highest MAE): Problems 30-34 (0.45164)


# DIAGNOSTICS: Reward Signal & Learning Quality Analysis

1. **Reward Signal Strength** - Is the reward varied enough to learn from?
2. **Action Imbalance** - Do different actions receive different rewards?
3. **Reward-Success Correlation** - Does our reward signal actually capture student learning?
4. **Early vs Late Training** - Does the model improve over time?
5. **State Distribution** - Is the state space diverse enough for learning?

**What to look for**: Red flags include uniform reward values, actions with identical mean rewards, weak correlations, or state clustering.

In [79]:
print("="*70)
print("DIAGNOSIS 1: REWARD SIGNAL STRENGTH")
print("="*70)
print("\nWhat to look for:")
print("  • Reward should span a wide range (e.g., -1 to +1, not all 0.15±0.02)")
print("  • High std dev indicates the model has signal to learn from")
print("  • If all rewards cluster near 0, the agent has no incentive to choose actions differently")
print("\nReward Statistics:")
print(f"  Count: {len(df_memory)}")
print(f"  Mean: {df_memory['reward'].mean():.4f}")
print(f"  Std Dev: {df_memory['reward'].std():.4f}")
print(f"  Min: {df_memory['reward'].min():.4f}")
print(f"  Max: {df_memory['reward'].max():.4f}")
print(f"  Median: {df_memory['reward'].median():.4f}")
print(f"  25th percentile: {df_memory['reward'].quantile(0.25):.4f}")
print(f"  75th percentile: {df_memory['reward'].quantile(0.75):.4f}")

# Histogram of rewards
fig = go.Figure(data=[
    go.Histogram(
        x=df_memory['reward'],
        nbinsx=30,
        marker=dict(color='rgba(100, 150, 255, 0.7)', line=dict(color='darkblue', width=1)),
        name='Reward Distribution'
    )
])
fig.update_layout(
    title="Reward Signal Distribution (All Transitions)",
    xaxis_title="Reward Value",
    yaxis_title="Frequency",
    height=400,
    width=900,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=11)
)
fig.show()

print(f"\n✓ If histogram is narrow/single peak → WEAK SIGNAL (problem)")
print(f"✓ If histogram is wide/bimodal → GOOD SIGNAL (desired)")

DIAGNOSIS 1: REWARD SIGNAL STRENGTH

What to look for:
  • Reward should span a wide range (e.g., -1 to +1, not all 0.15±0.02)
  • High std dev indicates the model has signal to learn from
  • If all rewards cluster near 0, the agent has no incentive to choose actions differently

Reward Statistics:
  Count: 190
  Mean: 0.6843
  Std Dev: 0.3547
  Min: -0.2000
  Max: 1.4000
  Median: 0.7000
  25th percentile: 0.4000
  75th percentile: 0.7425



✓ If histogram is narrow/single peak → WEAK SIGNAL (problem)
✓ If histogram is wide/bimodal → GOOD SIGNAL (desired)


In [80]:
print("\n" + "="*70)
print("DIAGNOSIS 2: ACTION BALANCE & MEAN REWARDS BY ACTION")
print("="*70)
print("\nWhat to look for:")
print("  • Actions should be balanced (don't want 80% one action, 10% each other)")
print("  • Mean rewards should DIFFER between actions (e.g., A=0.2, B=0.1, C=0.05)")
print("  • If all actions have nearly identical mean rewards → model can't learn to prefer one")
print("  • Large std dev within action → high variance, harder to learn\n")

ACTIONS = ["SOCRATIC_PROBE", "CONCEPTUAL_HINT", "STRUCTURAL_SCAFFOLD"]
# Action distribution and mean rewards
action_stats = df_memory.groupby('action').agg({
    'reward': ['count', 'mean', 'std', 'min', 'max'],
    'student_success': 'mean',
    'pedagogical_quality': 'mean'
}).round(4)
print("Action Statistics:")
print(action_stats)

# Calculate percentages
action_counts_total = df_memory['action'].value_counts()
print("\nAction Counts & Percentages:")
for action in ACTIONS:
    count = action_counts_total.get(action, 0)
    pct = (count / len(df_memory)) * 100
    mean_reward = df_memory[df_memory['action'] == action]['reward'].mean()
    print(f"  {action:20s}: {count:4d} ({pct:5.1f}%)  |  mean_reward={mean_reward:7.4f}")

# Visualize action rewards with error bars
action_reward_stats = df_memory.groupby('action')['reward'].agg(['mean', 'std', 'count']).reindex(ACTIONS)

fig = go.Figure(data=[
    go.Bar(
        x=action_reward_stats.index,
        y=action_reward_stats['mean'],
        error_y=dict(
            type='data',
            array=action_reward_stats['std'],
            visible=True
        ),
        marker=dict(
            color=['rgba(100, 149, 237, 0.8)', 'rgba(144, 238, 144, 0.8)', 'rgba(255, 165, 0, 0.8)'],
            line=dict(color='black', width=1.5)
        ),
        text=[f"n={int(c)}" for c in action_reward_stats['count']],
        textposition='outside',
        hovertemplate='<b>%{x}</b><br>Mean Reward: %{y:.4f}<br>Std: %{error_y.array:.4f}<extra></extra>'
    )
])

fig.update_layout(
    title="Mean Reward by Pedagogical Action (with ±1 Std Dev)",
    xaxis_title="Pedagogical Action",
    yaxis_title="Mean Reward",
    height=400,
    width=800,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=12),
    showlegend=False
)
fig.show()

print(f"\n✓ If all means are similar (within 0.01) → WEAK SIGNAL (problem)")
print(f"✓ If means differ by >0.05 → GOOD SIGNAL (desired)")
print(f"✓ If one action >70% of data → IMBALANCE (exploration too low?)")


DIAGNOSIS 2: ACTION BALANCE & MEAN REWARDS BY ACTION

What to look for:
  • Actions should be balanced (don't want 80% one action, 10% each other)
  • Mean rewards should DIFFER between actions (e.g., A=0.2, B=0.1, C=0.05)
  • If all actions have nearly identical mean rewards → model can't learn to prefer one
  • Large std dev within action → high variance, harder to learn

Action Statistics:
                    reward                           student_success  \
                     count    mean     std  min  max            mean   
action                                                                 
CONCEPTUAL_HINT         65  0.7379  0.3949 -0.2  1.4          0.3882   
SOCRATIC_PROBE          63  0.6156  0.2956 -0.2  1.4          0.1129   
STRUCTURAL_SCAFFOLD     62  0.6980  0.3596  0.1  1.4          0.2460   

                    pedagogical_quality  
                                   mean  
action                                   
CONCEPTUAL_HINT                  1.3077  
SO


✓ If all means are similar (within 0.01) → WEAK SIGNAL (problem)
✓ If means differ by >0.05 → GOOD SIGNAL (desired)
✓ If one action >70% of data → IMBALANCE (exploration too low?)


In [81]:
print("\n" + "="*70)
print("DIAGNOSIS 3: REWARD-SUCCESS CORRELATION")
print("="*70)
print("\nWhat to look for:")
print("  • Reward SHOULD correlate with student_success (higher code success = higher reward)")
print("  • Correlation coefficient should be >0.3 (moderate) or >0.5 (strong)")
print("  • If correlation is 0 or negative → reward signal is not capturing learning\n")

# Compute correlations
corr_reward_success = df_memory['reward'].corr(df_memory['student_success'])
corr_reward_pedagogy = df_memory['reward'].corr(df_memory['pedagogical_quality'])
corr_success_pedagogy = df_memory['student_success'].corr(df_memory['pedagogical_quality'])

print(f"Correlation: reward ↔ student_success: {corr_reward_success:.4f}")
print(f"Correlation: reward ↔ pedagogical_quality: {corr_reward_pedagogy:.4f}")
print(f"Correlation: student_success ↔ pedagogical_quality: {corr_success_pedagogy:.4f}")

# Scatter: student_success vs reward
fig = go.Figure(data=[
    go.Scatter(
        x=df_memory['student_success'],
        y=df_memory['reward'],
        mode='markers',
        marker=dict(
            size=6,
            color=df_memory['i'],  # Color by turn index
            colorscale='Viridis',
            colorbar=dict(title="Turn (i)"),
            line=dict(color='white', width=0.5),
            opacity=0.7
        ),
        text=[f"Problem {p}, Turn {i}" for p, i in zip(df_memory['problem'], df_memory['i'])],
        hovertemplate='<b>Turn %{text}</b><br>Student Success: %{x:.2%}<br>Reward: %{y:.4f}<extra></extra>'
    )
])

fig.update_layout(
    title=f"Reward vs Student Success (Correlation: {corr_reward_success:.3f})",
    xaxis_title="Student Code Success Rate (% tests passed)",
    yaxis_title="Reward Signal",
    height=500,
    width=900,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=11),
    xaxis=dict(tickformat='.0%')
)
fig.show()

# Scatter: pedagogical_quality vs reward
fig = go.Figure(data=[
    go.Scatter(
        x=df_memory['pedagogical_quality'],
        y=df_memory['reward'],
        mode='markers',
        marker=dict(
            size=6,
            color=df_memory['student_level'],
            colorscale='Plasma',
            colorbar=dict(title="Student Level"),
            line=dict(color='white', width=0.5),
            opacity=0.7
        ),
        text=[f"Problem {p}, Turn {i}" for p, i in zip(df_memory['problem'], df_memory['i'])],
        hovertemplate='<b>Turn %{text}</b><br>Pedagogical Quality: %{x:.2f}<br>Reward: %{y:.4f}<extra></extra>'
    )
])

fig.update_layout(
    title=f"Reward vs Pedagogical Quality (Correlation: {corr_reward_pedagogy:.3f})",
    xaxis_title="Pedagogical Quality Score",
    yaxis_title="Reward Signal",
    height=500,
    width=900,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=11)
)
fig.show()

print(f"\n✓ If correlations <0.1 → Your reward may not capture what you want (problem)")
print(f"✓ If correlations >0.3 → GOOD SIGNAL (desired)")


DIAGNOSIS 3: REWARD-SUCCESS CORRELATION

What to look for:
  • Reward SHOULD correlate with student_success (higher code success = higher reward)
  • Correlation coefficient should be >0.3 (moderate) or >0.5 (strong)
  • If correlation is 0 or negative → reward signal is not capturing learning

Correlation: reward ↔ student_success: 0.7867
Correlation: reward ↔ pedagogical_quality: 0.5352
Correlation: student_success ↔ pedagogical_quality: -0.0854



✓ If correlations <0.1 → Your reward may not capture what you want (problem)
✓ If correlations >0.3 → GOOD SIGNAL (desired)


In [82]:
print("\n" + "="*70)
print("DIAGNOSIS 4: EARLY VS LATE TRAINING COMPARISON")
print("="*70)
print("\nWhat to look for:")
print("  • Early problems (0-25): Baseline performance (model learning from scratch)")
print("  • Late problems (75-99): Should show improvement IF learning is working")
print("  • Compare: mean reward, accuracy, action distribution, model MAE")
print("  • If early ≈ late → Model NOT learning from experience\n")

# Split into early, mid, late
early_idx = df_memory['problem'] < 25
mid_idx = (df_memory['problem'] >= 25) & (df_memory['problem'] < 75)
late_idx = df_memory['problem'] >= 75

early_data = df_memory[early_idx]
mid_data = df_memory[mid_idx]
late_data = df_memory[late_idx]

print("EARLY PROBLEMS (0-24):")
print(f"  Transitions: {len(early_data)}")
print(f"  Mean reward: {early_data['reward'].mean():.4f}")
print(f"  Mean accuracy: {early_data['student_success'].mean():.2%}")
print(f"  Mean pedagogical quality: {early_data['pedagogical_quality'].mean():.4f}")

print("\nMID PROBLEMS (25-74):")
print(f"  Transitions: {len(mid_data)}")
print(f"  Mean reward: {mid_data['reward'].mean():.4f}")
print(f"  Mean accuracy: {mid_data['student_success'].mean():.2%}")
print(f"  Mean pedagogical quality: {mid_data['pedagogical_quality'].mean():.4f}")

print("\nLATE PROBLEMS (75-99):")
print(f"  Transitions: {len(late_data)}")
print(f"  Mean reward: {late_data['reward'].mean():.4f}")
print(f"  Mean accuracy: {late_data['student_success'].mean():.2%}")
print(f"  Mean pedagogical quality: {late_data['pedagogical_quality'].mean():.4f}")

# Calculate improvements
reward_improvement = late_data['reward'].mean() - early_data['reward'].mean()
accuracy_improvement = late_data['student_success'].mean() - early_data['student_success'].mean()

print(f"\nCHANGE (Late - Early):")
print(f"  Reward: {reward_improvement:+.4f}")
print(f"  Accuracy: {accuracy_improvement:+.2%}")

# Visualization: Rolling mean of key metrics
df_sorted = df_memory.sort_values('problem').reset_index(drop=True)
window = max(10, len(df_sorted) // 20)  # Adaptive window size

rolling_reward = df_sorted['reward'].rolling(window=window, center=True).mean()
rolling_accuracy = df_sorted['student_success'].rolling(window=window, center=True).mean()

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=("Mean Reward Over Training", "Student Accuracy Over Training"),
    vertical_spacing=0.12
)

fig.add_trace(
    go.Scatter(
        x=df_sorted['problem'],
        y=rolling_reward,
        mode='lines',
        name='Reward (rolling avg)',
        line=dict(color='orange', width=3),
        hovertemplate='Problem %{x}: Reward = %{y:.4f}<extra></extra>'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=df_sorted['problem'],
        y=rolling_accuracy,
        mode='lines',
        name='Accuracy (rolling avg)',
        line=dict(color='green', width=3),
        hovertemplate='Problem %{x}: Accuracy = %{y:.2%}<extra></extra>'
    ),
    row=2, col=1
)

fig.update_xaxes(title_text="Problem Index (0-99)", row=2, col=1)
fig.update_yaxes(title_text="Mean Reward", row=1, col=1)
fig.update_yaxes(title_text="Mean Accuracy", row=2, col=1)

fig.update_layout(
    title_text=f"Training Progress: Rolling Average (window={window})",
    height=600,
    width=1000,
    hovermode='x unified',
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=11)
)
fig.show()

print(f"\n✓ If both lines are flat/noisy → Model NOT learning (problem)")
print(f"✓ If late > early clearly → LEARNING DETECTED (desired)")


DIAGNOSIS 4: EARLY VS LATE TRAINING COMPARISON

What to look for:
  • Early problems (0-25): Baseline performance (model learning from scratch)
  • Late problems (75-99): Should show improvement IF learning is working
  • Compare: mean reward, accuracy, action distribution, model MAE
  • If early ≈ late → Model NOT learning from experience

EARLY PROBLEMS (0-24):
  Transitions: 74
  Mean reward: 0.8102
  Mean accuracy: 39.10%
  Mean pedagogical quality: 1.5405

MID PROBLEMS (25-74):
  Transitions: 116
  Mean reward: 0.6040
  Mean accuracy: 16.09%
  Mean pedagogical quality: 1.3966

LATE PROBLEMS (75-99):
  Transitions: 0
  Mean reward: nan
  Mean accuracy: nan%
  Mean pedagogical quality: nan

CHANGE (Late - Early):
  Reward: +nan
  Accuracy: +nan%



✓ If both lines are flat/noisy → Model NOT learning (problem)
✓ If late > early clearly → LEARNING DETECTED (desired)


In [83]:
print("\n" + "="*70)
print("DIAGNOSIS 5: STATE SPACE DISTRIBUTION")
print("="*70)
print("\nWhat to look for:")
print("  • Turn index (i): Should be distributed 0-9 (more 0-2 normal due to early stopping)")
print("  • Student/Tutor levels: Should span 1-4 to cover diverse scenarios")
print("  • If all states cluster in one corner → Model can't learn diverse behaviors\n")

# Turn distribution
print("TURN INDEX DISTRIBUTION (i):")
turn_counts = df_memory['i'].value_counts().sort_index()
for turn, count in turn_counts.items():
    pct = (count / len(df_memory)) * 100
    bar = '█' * int(pct / 2)
    print(f"  Turn {turn}: {count:4d} ({pct:5.1f}%) {bar}")

# Level distributions
print("\nSTUDENT LEVEL DISTRIBUTION (1-4 scale):")
level_counts = df_memory['last_student_level'].value_counts().sort_index()
for level, count in level_counts.items():
    pct = (count / len(df_memory)) * 100
    bar = '█' * int(pct / 2)
    print(f"  Level {level}: {count:4d} ({pct:5.1f}%) {bar}")

print("\nTUTOR LEVEL DISTRIBUTION (1-4 scale):")
level_counts = df_memory['last_tutor_level'].value_counts().sort_index()
for level, count in level_counts.items():
    pct = (count / len(df_memory)) * 100
    bar = '█' * int(pct / 2)
    print(f"  Level {level}: {count:4d} ({pct:5.1f}%) {bar}")

# Visualizations
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Turn Index Distribution", "Student Level Distribution",
                    "Tutor Level Distribution", "(Student Level, Tutor Level) Heatmap"),
    specs=[[{"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "heatmap"}]]
)

# Turn distribution
turn_data = df_memory['i'].value_counts().sort_index()
fig.add_trace(
    go.Bar(x=turn_data.index, y=turn_data.values, marker=dict(color='rgba(100, 150, 255, 0.7)'),
           name='Turn Count', hovertemplate='Turn %{x}: %{y}<extra></extra>'),
    row=1, col=1
)

# Student level distribution
student_level_data = df_memory['last_student_level'].value_counts().sort_index()
fig.add_trace(
    go.Bar(x=student_level_data.index, y=student_level_data.values, marker=dict(color='rgba(144, 238, 144, 0.7)'),
           name='Student Level Count', hovertemplate='Level %{x}: %{y}<extra></extra>'),
    row=1, col=2
)

# Tutor level distribution
tutor_level_data = df_memory['last_tutor_level'].value_counts().sort_index()
fig.add_trace(
    go.Bar(x=tutor_level_data.index, y=tutor_level_data.values, marker=dict(color='rgba(255, 165, 0, 0.7)'),
           name='Tutor Level Count', hovertemplate='Level %{x}: %{y}<extra></extra>'),
    row=2, col=1
)

# Cross-tabulation heatmap
state_heatmap = pd.crosstab(df_memory['last_student_level'], df_memory['last_tutor_level'])
fig.add_trace(
    go.Heatmap(z=state_heatmap.values, x=state_heatmap.columns, y=state_heatmap.index,
               colorscale='Blues', name='Count',
               hovertemplate='Student %{y} → Tutor %{x}: %{z}<extra></extra>'),
    row=2, col=2
)

fig.update_xaxes(title_text="Turn Index (i)", row=1, col=1)
fig.update_xaxes(title_text="Student Level", row=1, col=2)
fig.update_xaxes(title_text="Tutor Level", row=2, col=1)
fig.update_xaxes(title_text="Tutor Level", row=2, col=2)

fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=2)
fig.update_yaxes(title_text="Count", row=2, col=1)
fig.update_yaxes(title_text="Student Level", row=2, col=2)

fig.update_layout(
    title_text="State Space Distribution Analysis",
    height=700,
    width=1200,
    plot_bgcolor='rgba(240, 240, 240, 0.5)',
    font=dict(size=10),
    showlegend=False
)
fig.show()

print(f"\n✓ If all distributions cluster in one region → STATE COLLAPSE (problem)")
print(f"✓ If distributions span 1-4 with variety → DIVERSE STATE SPACE (desired)")
print(f"✓ More turns 0-2 than 3-9 is EXPECTED (early stopping is normal)")


DIAGNOSIS 5: STATE SPACE DISTRIBUTION

What to look for:
  • Turn index (i): Should be distributed 0-9 (more 0-2 normal due to early stopping)
  • Student/Tutor levels: Should span 1-4 to cover diverse scenarios
  • If all states cluster in one corner → Model can't learn diverse behaviors

TURN INDEX DISTRIBUTION (i):
  Turn 0:   46 ( 24.2%) ████████████
  Turn 1:   35 ( 18.4%) █████████
  Turn 2:   25 ( 13.2%) ██████
  Turn 3:   17 (  8.9%) ████
  Turn 4:   16 (  8.4%) ████
  Turn 5:   12 (  6.3%) ███
  Turn 6:   12 (  6.3%) ███
  Turn 7:   10 (  5.3%) ██
  Turn 8:    9 (  4.7%) ██
  Turn 9:    8 (  4.2%) ██

STUDENT LEVEL DISTRIBUTION (1-4 scale):
  Level -1:   46 ( 24.2%) ████████████
  Level 1:    4 (  2.1%) █
  Level 2:   38 ( 20.0%) ██████████
  Level 3:   60 ( 31.6%) ███████████████
  Level 4:   42 ( 22.1%) ███████████

TUTOR LEVEL DISTRIBUTION (1-4 scale):
  Level -1:   46 ( 24.2%) ████████████
  Level 2:   45 ( 23.7%) ███████████
  Level 3:   41 ( 21.6%) ██████████
  Level 4:


✓ If all distributions cluster in one region → STATE COLLAPSE (problem)
✓ If distributions span 1-4 with variety → DIVERSE STATE SPACE (desired)
✓ More turns 0-2 than 3-9 is EXPECTED (early stopping is normal)


In [84]:
print("\n" + "="*70)
print("DIAGNOSIS 6: REWARD MODEL PREDICTION QUALITY")
print("="*70)
print("\nWhat to look for:")
print("  • Model MAE: Lower is better (model predicting rewards accurately)")
print("  • If MAE >> reward std → Predictions are poor")
print("  • If MAE << reward std → GOOD PREDICTIONS (desired)")
print("  • Check if predictions improve over time\n")

# Compute MAE for predictions
pred_columns_list = ['pred_reward_SOCRATIC_PROBE', 'pred_reward_CONCEPTUAL_HINT', 'pred_reward_STRUCTURAL_SCAFFOLD']
df_with_preds = df_memory[pred_columns_list + ['reward']].dropna()

if len(df_with_preds) > 0:
    # For each row, use the prediction of the action that was actually taken
    def get_predicted_reward_for_action(row):
        action = df_memory.loc[row.name, 'action']
        pred_col = f'pred_reward_{action}'
        if pred_col in df_with_preds.columns:
            return df_with_preds.loc[row.name, pred_col]
        return None
    
    # Simpler approach: use max prediction for each row
    best_pred = df_with_preds[pred_columns_list].max(axis=1)
    mae = (best_pred - df_with_preds['reward']).abs().mean()
    
    reward_std = df_memory['reward'].std()
    ratio = mae / reward_std if reward_std > 0 else float('inf')
    
    print(f"Rows with predictions: {len(df_with_preds)} / {len(df_memory)}")
    print(f"Mean Absolute Error (MAE): {mae:.6f}")
    print(f"Reward Std Dev: {reward_std:.6f}")
    print(f"MAE / Reward Std Ratio: {ratio:.4f}")
    
    if ratio > 1.0:
        print(f"⚠ HIGH RATIO: Model predictions are worse than random guessing!")
    elif ratio > 0.5:
        print(f"⚠ MODERATE RATIO: Model predictions have room for improvement")
    else:
        print(f"✓ GOOD RATIO: Model is making reasonable predictions")
    
    # Plot MAE over training time
    df_with_mae = df_with_preds.copy()
    df_with_mae['problem'] = df_memory.loc[df_with_mae.index, 'problem'].values
    df_with_mae['mae'] = (best_pred - df_with_preds['reward']).abs()
    
    mae_by_problem = df_with_mae.groupby('problem')['mae'].mean()
    
    fig = go.Figure(data=[
        go.Scatter(
            x=mae_by_problem.index,
            y=mae_by_problem.values,
            mode='lines+markers',
            name='MAE',
            line=dict(color='red', width=2),
            marker=dict(size=6),
            hovertemplate='Problem %{x}: MAE = %{y:.6f}<extra></extra>'
        )
    ])
    
    fig.add_hline(y=mae, line_dash="dash", line_color="red", annotation_text=f"Mean MAE: {mae:.6f}")
    
    fig.update_layout(
        title="Reward Model MAE Over Training",
        xaxis_title="Problem Index (0-99)",
        yaxis_title="Mean Absolute Error",
        height=400,
        width=1000,
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        font=dict(size=11)
    )
    fig.show()
    
    print(f"\n✓ If MAE decreases over time → Model is improving")
    print(f"✓ If MAE is flat/high → Model not learning or insufficient data")
else:
    print("No predictions found in memory (cold start phase?)")


DIAGNOSIS 6: REWARD MODEL PREDICTION QUALITY

What to look for:
  • Model MAE: Lower is better (model predicting rewards accurately)
  • If MAE >> reward std → Predictions are poor
  • If MAE << reward std → GOOD PREDICTIONS (desired)
  • Check if predictions improve over time

Rows with predictions: 164 / 190
Mean Absolute Error (MAE): 0.333580
Reward Std Dev: 0.354716
MAE / Reward Std Ratio: 0.9404
⚠ MODERATE RATIO: Model predictions have room for improvement



✓ If MAE decreases over time → Model is improving
✓ If MAE is flat/high → Model not learning or insufficient data


## Diagnostic Summary & Next Steps

Based on the diagnostics above, here are the most likely issues and potential fixes:

### Issue 1: Weak Reward Signal
**Symptom**: All rewards cluster near 0.15-0.25, with low variance
- **Root cause**: Reward components (student_success, pedagogical_quality, code_runs) are all contributing small amounts
- **Fixes**:
  - Increase weighting of student_success (increase 1-LAMBDA from 0.3)
  - Add bonuses for reaching later turns (agent persisting helps)
  - Consider binary reward (solved vs not solved) instead of continuous

### Issue 2: Actions Have Identical Rewards
**Symptom**: SOCRATIC_PROBE ≈ CONCEPTUAL_HINT ≈ STRUCTURAL_SCAFFOLD (all ≈0.18)
- **Root cause**: Different pedagogical moves may not actually affect student outcomes in simulation
- **Fixes**:
  - Verify judge agents are correctly evaluating tutor behavior
  - Add explicit reward bonus for actions that match student level
  - Increase sensitivity of pedagogy quality metric to action type

### Issue 3: Poor Correlation (Reward ↔ Student Success)
**Symptom**: Scatter plot is random cloud, correlation < 0.1
- **Root cause**: Reward formula doesn't capture learning
- **Fixes**:
  - Simplify reward: `reward = student_success + leakage_penalty`
  - Focus on cumulative progress, not turn-by-turn success
  - Add larger penalties for bad outcomes (leakage, changing problem)

### Issue 4: No Learning Curve (Early ≈ Late)
**Symptom**: Mean reward flat across problems 0-99
- **Root cause**: Insufficient training data or model can't learn from state features
- **Fixes**:
  - Increase retrain frequency vs data volume
  - Add interaction features (e.g., student_level × tutor_level)
  - Add more context to state (e.g., problem difficulty from LeetCode)
  - Lower epsilon exploration to let learned policy take effect

### Issue 5: State Space Collapse
**Symptom**: All states cluster at (student_level=1, tutor_level=1, i=0-2)
- **Root cause**: Early stopping dominates, or judges not calibrated
- **Fixes**:
  - Adjust stop conditions (let conversations run longer)
  - Recalibrate judge prompts for more varied level assignments
  - Add problem difficulty to state representation

### Issue 6: Model MAE Too High
**Symptom**: MAE >> reward_std (predictions worse than baseline)
- **Root cause**: Insufficient data or RF overfitting
- **Fixes**:
  - Use simpler model (linear regression instead of RF)
  - Add regularization to RF
  - Collect more data (train longer)
  - Feature engineering: normalize state features

In [85]:
## Feature importance

## Reward Model Feature Importance Analysis

In [86]:
import pickle

print("="*70)
print("REWARD MODEL ANALYSIS: FEATURE IMPORTANCE")
print("="*70)
print("\nWhat this shows:")
print("  • Which state features have the most influence on predicted rewards")
print("  • Higher importance = feature is more useful for decision-making")
print("  • If all importances are equal → model treats all features as useless\n")

# Load the trained reward model
model_path = RUN_PATH / "checkpoints" / "reward_model.pkl"

if model_path.exists():
    with open(model_path, 'rb') as f:
        reward_model = pickle.load(f)
    
    print(f"✓ Loaded reward model from {model_path}")
    
    # The model is a Pipeline with preprocessing and Random Forest
    # Get the Random Forest estimator
    rf_model = reward_model.named_steps['reg']
    
    # Get feature importances
    importances = rf_model.feature_importances_
    
    # Get feature names from the preprocessor
    preprocessor = reward_model.named_steps['pre']
    
    # After OneHotEncoder on categorical features, we need to get the feature names
    # Categorical features: last_action (3 values), action (3 values) = 6 one-hot features
    # Numerical features: i, last_student_level, last_tutor_level, last_reward = 4 features
    
    categorical_names = []
    for i, cat in enumerate(preprocessor.named_transformers_['cat'].get_feature_names_out()):
        categorical_names.append(f"cat_{cat}")
    
    # numerical_names = ['i', 'last_student_level', 'last_tutor_level', 'last_reward']
    numerical_names =  ["i", "last_student_level", "last_tutor_level", "last_reward", "last_coding_score"]
    
    all_feature_names = categorical_names + numerical_names
    
    print(f"\nTotal features in model: {len(all_feature_names)}")
    print(f"  Categorical (one-hot encoded): {len(categorical_names)}")
    print(f"  Numerical: {len(numerical_names)}")
    
    # Create DataFrame for easier analysis
    feature_importance_df = pd.DataFrame({
        'Feature': all_feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    print(f"\nFeature Importances (sorted by importance):")
    print(feature_importance_df.to_string(index=False))
    
    # Sum importances by feature type
    print(f"\nImportance by Type:")
    cat_importance = feature_importance_df[feature_importance_df['Feature'].str.startswith('cat_')]['Importance'].sum()
    num_importance = feature_importance_df[feature_importance_df['Feature'].str.startswith(('i', 'last_'))]['Importance'].sum()
    print(f"  Categorical features: {cat_importance:.4f}")
    print(f"  Numerical features: {num_importance:.4f}")
    
    # Visualization: Bar chart of feature importances
    fig = go.Figure(data=[
        go.Bar(
            x=feature_importance_df['Feature'],
            y=feature_importance_df['Importance'],
            marker=dict(
                color=feature_importance_df['Importance'],
                colorscale='Viridis',
                line=dict(color='black', width=1)
            ),
            text=[f"{v:.4f}" for v in feature_importance_df['Importance']],
            textposition='outside',
            hovertemplate='<b>%{x}</b><br>Importance: %{y:.6f}<extra></extra>'
        )
    ])
    
    fig.update_layout(
        title="Reward Model Feature Importances (Random Forest)",
        xaxis_title="Feature",
        yaxis_title="Importance Score",
        height=500,
        width=1000,
        plot_bgcolor='rgba(240, 240, 240, 0.5)',
        font=dict(size=11),
        xaxis=dict(tickangle=-45),
        showlegend=False
    )
    
    fig.show()
    
    # Interpretation
    top_feature = feature_importance_df.iloc[0]
    print(f"\n✓ Top feature: {top_feature['Feature']} (importance: {top_feature['Importance']:.4f})")
    print(f"✓ Model uses {'categorical' if cat_importance > num_importance else 'numerical'} features more")
    
    if feature_importance_df['Importance'].max() < 0.15:
        print("⚠ WARNING: All features have low importance → Model may not be learning meaningful patterns")
else:
    print(f"✗ Model file not found at {model_path}")
    print("  Make sure the training has completed and model was saved")

REWARD MODEL ANALYSIS: FEATURE IMPORTANCE

What this shows:
  • Which state features have the most influence on predicted rewards
  • Higher importance = feature is more useful for decision-making
  • If all importances are equal → model treats all features as useless

✓ Loaded reward model from outputs\20260315_gpt_4_1_beginner_test1\checkpoints\reward_model.pkl

Total features in model: 15
  Categorical (one-hot encoded): 10
  Numerical: 5

Feature Importances (sorted by importance):
                            Feature  Importance
                        last_reward    0.179847
                                  i    0.178639
                 last_student_level    0.106771
                  last_coding_score    0.102889
         cat_action_CONCEPTUAL_HINT    0.066979
        cat_problem_difficulty_Easy    0.051668
     cat_action_STRUCTURAL_SCAFFOLD    0.050404
                   last_tutor_level    0.050188
        cat_problem_difficulty_Hard    0.037485
          cat_action_SOCRATIC


✓ Top feature: last_reward (importance: 0.1798)
✓ Model uses numerical features more
